In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

# FLenQA linear probe evaluation

This notebook validates the frozen probe assets, evaluates them on held-out problems at every context length, and compares answer decodability with the saved model answers. Probes and thresholds are never refit on test data.


In [ ]:
%pip install -qq --disable-pip-version-check pandas matplotlib

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import transformers
from datasets import load_from_disk
from sklearn.metrics import log_loss, roc_auc_score
from tqdm.auto import tqdm

from experiments.jlens_readout_sanity.constants import MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.environments.colab import initialize_colab
from jlens_reasoning.evaluation import evaluate_paper_binary

context = initialize_colab(enable_wandb=False, require_cuda=True)
ASSET_DIR = context.checkpoints_dir / "flenqa-probe-assets"
PROBE_PATH = ASSET_DIR / "probes.pt"
METADATA_PATH = ASSET_DIR / "metadata.json"
SPLIT_PATH = ASSET_DIR / "problem_split.json"
MODEL_OUTPUT_PATH = context.runs_dir / "flenqa-full-run" / "model_outputs.parquet"
EXPECTED_CONTEXT_SIZES = (250, 500, 1000, 2000, 3000)

## Load and sanity-check the frozen assets

The split is inherited from the training notebook. The saved validation metrics, not test performance, will choose the headline layer.

In [ ]:
assert PROBE_PATH.is_file() and METADATA_PATH.is_file() and SPLIT_PATH.is_file()
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
split_asset = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
checkpoint = torch.load(PROBE_PATH, map_location="cpu", weights_only=False)
assert metadata["format_version"] == 1 == checkpoint["format_version"]
assert metadata["split"] == split_asset
assert metadata["context_sizes"] == [250, 500]
assert "test" not in metadata["example_counts"]
assert all("test" not in metrics for metrics in metadata["probe_metrics"].values())

split_problem_ids = split_asset["problems"]
train_ids = set(split_problem_ids["train"])
validation_ids = set(split_problem_ids["validation"])
test_ids = set(split_problem_ids["test"])
assert len(train_ids) == 180 and len(validation_ids) == 60 and len(test_ids) == 60
assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)
assert len(train_ids | validation_ids | test_ids) == 300

causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()
num_layers = int(causal_lm.config.num_hidden_layers)
hidden_dim = int(causal_lm.config.hidden_size)
assert metadata["num_layers"] == num_layers
assert metadata["hidden_dim"] == hidden_dim
assert set(checkpoint["layers"]) == set(range(num_layers))
for layer in range(num_layers):
    asset = checkpoint["layers"][layer]
    for name in ("weight", "unit_weight", "training_mean"):
        tensor = asset[name].float()
        assert tensor.shape == (hidden_dim,)
        assert torch.isfinite(tensor).all()
    assert asset["bias"].ndim == 0 and torch.isfinite(asset["bias"])
    assert torch.isclose(
        torch.linalg.vector_norm(asset["unit_weight"].float()),
        torch.tensor(1.0),
        atol=1e-5,
    )

print(
    {
        "layers": num_layers,
        "hidden_dim": hidden_dim,
        "split_sizes": {key: len(value) for key, value in split_problem_ids.items()},
        "fit_context_sizes": metadata["context_sizes"],
    }
)

## Evaluate the frozen probes on held-out problems

The probe was fit on 250/500-token prompts only. Applying it to 1000/2000/3000-token prompts is an intentional out-of-distribution test of whether the short-context answer direction remains decodable.

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)
test_rows = [row for row in rows if row.problem_id in test_ids]
assert test_rows and {row.problem_id for row in test_rows} == test_ids
test_prompts = prepare_prompts(test_rows)
prompt_context_sizes = {}
for prompt in test_prompts:
    context_sizes = {item.ctx_size for item in prompt.provenance}
    assert len(context_sizes) == 1
    prompt_context_sizes[prompt.prompt_id] = context_sizes.pop()
assert set(prompt_context_sizes.values()) == set(EXPECTED_CONTEXT_SIZES)

model_outputs = pq.read_table(MODEL_OUTPUT_PATH).to_pylist()
model_records = {record["prompt_id"]: record for record in model_outputs}
assert len(model_records) == len(model_outputs)
assert set(prompt_context_sizes).issubset(model_records)
assert all(record["model_name"] == MODEL_NAME for record in model_records.values())

test_examples = []
for prompt in test_prompts:
    record = model_records[prompt.prompt_id]
    assert record["problem_id"] == prompt.problem_id
    assert record["label"] == prompt.label
    model_evaluation = evaluate_paper_binary(
        record["generated_text"], expected=prompt.label
    )
    test_examples.append(
        {
            "prompt_id": prompt.prompt_id,
            "problem_id": prompt.problem_id,
            "task": prompt.task,
            "ctx_size": prompt_context_sizes[prompt.prompt_id],
            "label": int(prompt.label),
            "model_answer": model_evaluation.verdict,
            "model_correct": model_evaluation.correct,
        }
    )
assert {example["ctx_size"] for example in test_examples} == set(EXPECTED_CONTEXT_SIZES)
pd.DataFrame(test_examples).groupby("ctx_size").size()

In [ ]:
layer_features = [[] for _ in range(num_layers)]
for prompt in tqdm(test_prompts, desc="Extracting held-out states", unit="prompt"):
    encoded = tokenizer(prompt.text, return_tensors="pt", truncation=False)
    input_ids = encoded["input_ids"].to(context.device)
    attention_mask = encoded["attention_mask"].to(context.device)
    assert input_ids.ndim == 2 and input_ids.shape[0] == 1
    assert 0 < input_ids.shape[1] <= 4096
    with torch.inference_mode():
        outputs = causal_lm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    assert outputs.hidden_states is not None
    assert len(outputs.hidden_states) == num_layers + 1
    for layer in range(num_layers):
        layer_features[layer].append(
            outputs.hidden_states[layer + 1][0, -1, :]
            .detach()
            .to("cpu", dtype=torch.float32)
        )
layer_features = [torch.stack(features) for features in layer_features]
assert all(
    features.shape == (len(test_prompts), hidden_dim) for features in layer_features
)

In [ ]:
test_labels = torch.tensor(
    [example["label"] for example in test_examples], dtype=torch.float32
)
layer_results = []
layer_scores = {}
for layer in range(num_layers):
    asset = checkpoint["layers"][layer]
    scores = (
        torch.mv(
            layer_features[layer] - asset["training_mean"].float(),
            asset["weight"].float(),
        )
        + asset["bias"].float()
    )
    assert torch.isfinite(scores).all()
    probabilities = torch.sigmoid(scores).numpy()
    predictions = (scores > 0).numpy()
    gold_margins = torch.where(test_labels == 1, scores, -scores)
    gold_probabilities = torch.sigmoid(gold_margins).numpy()
    layer_scores[layer] = {
        "score": scores,
        "gold_margin": gold_margins,
        "gold_probability": gold_probabilities,
        "prediction": predictions,
    }
    layer_results.append(
        {
            "layer": layer,
            "validation_accuracy": metadata["probe_metrics"][str(layer)]["validation"][
                "accuracy"
            ],
            "validation_log_loss": metadata["probe_metrics"][str(layer)]["validation"][
                "log_loss"
            ],
            "test_accuracy": float((predictions == test_labels.numpy()).mean()),
            "test_log_loss": float(log_loss(test_labels.numpy(), probabilities)),
        }
    )

layer_table = pd.DataFrame(layer_results)
headline_layer = int(
    layer_table.sort_values(["validation_log_loss", "layer"]).iloc[0]["layer"]
)
headline = layer_scores[headline_layer]
display(layer_table)
print(f"Headline layer selected by validation log loss only: {headline_layer}")

## Probe accuracy versus context length

Gold-aligned margins ask whether the correct answer remains represented, even when the model’s generated answer is wrong.

In [ ]:
headline_frame = pd.DataFrame(test_examples)
headline_frame["probe_score"] = headline["score"].numpy()
headline_frame["gold_margin"] = headline["gold_margin"].numpy()
headline_frame["gold_probability"] = headline["gold_probability"]
headline_frame["probe_prediction"] = headline["prediction"]
headline_frame["probe_correct"] = (
    headline_frame["probe_prediction"] == headline_frame["label"]
)
context_summary = (
    headline_frame.groupby("ctx_size", as_index=False)
    .agg(
        test_examples=("problem_id", "size"),
        probe_accuracy=("probe_correct", "mean"),
        mean_gold_probability=("gold_probability", "mean"),
        mean_gold_margin=("gold_margin", "mean"),
        model_accuracy=("model_correct", "mean"),
    )
    .sort_values("ctx_size")
)
display(context_summary)
plt.figure(figsize=(7, 4))
plt.plot(
    context_summary["ctx_size"],
    context_summary["model_accuracy"],
    marker="o",
    label="model",
)
plt.plot(
    context_summary["ctx_size"],
    context_summary["probe_accuracy"],
    marker="o",
    label="probe",
)
plt.xticks(EXPECTED_CONTEXT_SIZES)
plt.ylim(0, 1)
plt.xlabel("Nominal context size")
plt.ylabel("Accuracy")
plt.title(f"Held-out accuracy: model vs. probe (layer {headline_layer})")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## Model-versus-probe failure cases

A correct probe prediction at long context means the short-context answer direction is still decodable. A wrong probe prediction alone does not show that the information disappeared.

## Raw probe-score distributions

These histograms use the frozen validation-selected layer's raw score, not its sigmoid output. Shared x-axis limits make label separation and shifts in the score location visible as context grows.

In [ ]:
score_bins = np.linspace(
    headline_frame["probe_score"].min(),
    headline_frame["probe_score"].max(),
    25,
)
fig, axes = plt.subplots(1, len(EXPECTED_CONTEXT_SIZES), figsize=(14, 3.5), sharex=True, sharey=True)
for axis, ctx_size in zip(axes, EXPECTED_CONTEXT_SIZES, strict=True):
    group = headline_frame[headline_frame["ctx_size"] == ctx_size]
    for label, color in [(True, "tab:blue"), (False, "tab:orange")]:
        axis.hist(
            group.loc[group["label"] == label, "probe_score"],
            bins=score_bins, density=True, alpha=0.5, color=color, label=str(label),
        )
    axis.axvline(0, color="black", linewidth=1)
    axis.set_title(f"context {ctx_size}")
    axis.set_xlabel("raw probe score")
axes[0].set_ylabel("density")
axes[-1].legend(title="gold")
fig.suptitle(f"Headline-layer probe scores (layer {headline_layer})")
plt.tight_layout()
plt.show()

In [ ]:
headline_frame["category"] = np.select(
    [
        headline_frame["model_correct"] & headline_frame["probe_correct"],
        headline_frame["model_correct"] & ~headline_frame["probe_correct"],
        ~headline_frame["model_correct"] & headline_frame["probe_correct"],
    ],
    [
        "model correct / probe correct",
        "model correct / probe wrong",
        "model wrong / probe correct",
    ],
    default="model wrong / probe wrong",
)
failure_counts = (
    headline_frame.groupby(["ctx_size", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=EXPECTED_CONTEXT_SIZES, fill_value=0)
)
failure_percentages = failure_counts.div(failure_counts.sum(axis=1), axis=0).round(3)
display(failure_counts)
display(failure_percentages)

interesting = headline_frame[
    (~headline_frame["model_correct"]) & headline_frame["probe_correct"]
].sort_values(["ctx_size", "gold_probability"], ascending=[False, False])
sample_columns = [
    "problem_id",
    "task",
    "ctx_size",
    "label",
    "model_answer",
    "probe_prediction",
    "gold_probability",
    "gold_margin",
]
display(interesting[sample_columns].head(8))

## Label and task sanity check

These groupings check whether the long-context probe-correct/model-wrong pattern is concentrated in one gold label or task.

In [ ]:
sanity_frame = headline_frame.assign(
    model_wrong=~headline_frame["model_correct"],
    model_wrong_probe_correct=(
        ~headline_frame["model_correct"] & headline_frame["probe_correct"]
    ),
)


def sanity_summary(group_columns):
    summary = sanity_frame.groupby(group_columns, as_index=False, observed=True).agg(
        examples=("problem_id", "size"),
        model_accuracy=("model_correct", "mean"),
        probe_accuracy=("probe_correct", "mean"),
        model_wrong=("model_wrong", "sum"),
        model_wrong_probe_correct=("model_wrong_probe_correct", "sum"),
    )
    summary["p_probe_correct_given_model_wrong"] = (
        summary["model_wrong_probe_correct"] / summary["model_wrong"]
    ).fillna(0.0)
    return summary


label_sanity = sanity_summary(["ctx_size", "label"])
task_label_sanity = sanity_summary(["ctx_size", "task", "label"])
display(label_sanity)
display(task_label_sanity)

## Problem-level failure summary

Prompt variants for one problem are correlated. This summary collapses them to problem flags so the model-wrong / probe-correct pattern is not counted as independent evidence for every variant.

In [ ]:
problem_flags = (
    sanity_frame.groupby(["ctx_size", "problem_id", "label"], as_index=False)
    .agg(
        model_wrong=("model_wrong", "any"),
        model_wrong_probe_correct=("model_wrong_probe_correct", "any"),
    )
)
problem_failure_summary = (
    sanity_frame.groupby("ctx_size", as_index=False)
    .agg(
        prompt_variants=("prompt_id", "size"),
        unique_problems=("problem_id", "nunique"),
        model_wrong_prompts=("model_wrong", "sum"),
        model_wrong_probe_correct_prompts=("model_wrong_probe_correct", "sum"),
    )
)
problem_counts = (
    problem_flags.groupby("ctx_size", as_index=False)
    .agg(
        unique_model_wrong_problems=("model_wrong", "sum"),
        unique_model_wrong_probe_correct_problems=("model_wrong_probe_correct", "sum"),
    )
)
problem_failure_summary = problem_failure_summary.merge(problem_counts, on="ctx_size")
display(problem_failure_summary)
problem_failure_by_label = (
    problem_flags.groupby(["ctx_size", "label"], as_index=False)
    .agg(
        unique_model_wrong_problems=("model_wrong", "sum"),
        unique_model_wrong_probe_correct_problems=("model_wrong_probe_correct", "sum"),
    )
)
display(problem_failure_by_label)

## Headline-layer AUROC by context length

AUROC uses the raw probe score, so it tests label separability without relying on the fixed zero threshold.

In [ ]:
headline_auroc = pd.DataFrame(
    [
        {
            "ctx_size": ctx_size,
            "auroc": roc_auc_score(group["label"], group["probe_score"]),
        }
        for ctx_size, group in headline_frame.groupby("ctx_size", sort=True)
    ]
).sort_values("ctx_size")
display(headline_auroc)

## Score drift versus label separation

The overall mean tracks score location, while the True-minus-False difference tracks answer-label separation. Neither quantity changes the frozen probe or its threshold.

In [ ]:
score_summary = (
    headline_frame.assign(
        true_score=headline_frame["probe_score"].where(headline_frame["label"] == 1),
        false_score=headline_frame["probe_score"].where(headline_frame["label"] == 0),
    )
    .groupby("ctx_size", as_index=False)
    .agg(
        mean_true_score=("true_score", "mean"),
        mean_false_score=("false_score", "mean"),
        overall_mean_score=("probe_score", "mean"),
    )
)
score_summary["score_separation"] = (
    score_summary["mean_true_score"] - score_summary["mean_false_score"]
)
score_summary = score_summary.merge(headline_auroc, on="ctx_size")
display(score_summary)
score_summary.set_index("ctx_size")[["overall_mean_score", "score_separation"]].plot(
    marker="o", figsize=(7, 4)
)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Nominal context size")
plt.ylabel("raw probe score")
plt.title("Score location and True/False separation")
plt.grid(alpha=0.25)
plt.show()

## AUROC across layers and context lengths

This descriptive view reuses the already extracted raw scores for every layer. It does not choose a new layer or tune a threshold on the test set.

In [ ]:
labels_np = headline_frame["label"].to_numpy()
contexts_np = headline_frame["ctx_size"].to_numpy()
all_layer_auroc = pd.DataFrame(
    [
        {
            "layer": layer,
            "ctx_size": ctx_size,
            "auroc": roc_auc_score(
                labels_np[contexts_np == ctx_size],
                layer_scores[layer]["score"].numpy()[contexts_np == ctx_size],
            ),
        }
        for layer in range(num_layers)
        for ctx_size in EXPECTED_CONTEXT_SIZES
    ]
)
all_layer_auroc_pivot = all_layer_auroc.pivot(
    index="layer", columns="ctx_size", values="auroc"
)
display(all_layer_auroc_pivot)
plt.figure(figsize=(8, 4.5))
for ctx_size, group in all_layer_auroc.groupby("ctx_size", sort=True):
    plt.plot(group["layer"], group["auroc"], marker="o", label=str(ctx_size))
plt.axhline(0.5, color="black", linestyle="--", linewidth=1)
plt.xlabel("Layer")
plt.ylabel("AUROC")
plt.title("Raw probe-score AUROC across layers")
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.legend(title="Context size")
plt.show()

## Interpretation

Probe accuracy at long context can be misleading because the fixed short-context decision threshold becomes biased toward label 1. AUROC separates the question of label separability from that threshold. At the headline layer, AUROC decreases with context length but remains above chance. The all-layer AUROC view checks whether this degradation is general across layers or whether the strongest representation shifts elsewhere. These results establish decodability only; they do not show that the model uses the information causally. J-Lens analysis belongs in the next notebook.

## Short summary

The summary reports descriptive held-out results only; it does not make a causal claim about why model accuracy changes.

In [ ]:
overall_probe_probability = torch.sigmoid(headline["score"]).numpy()
overall_probe_accuracy = float(headline_frame["probe_correct"].mean())
overall_model_accuracy = float(headline_frame["model_correct"].mean())
wrong_model_right_probe = headline_frame[
    (~headline_frame["model_correct"]) & headline_frame["probe_correct"]
]
long_context = headline_frame[headline_frame["ctx_size"] == max(EXPECTED_CONTEXT_SIZES)]
long_interesting = long_context[
    (~long_context["model_correct"]) & long_context["probe_correct"]
]
print(
    {
        "headline_layer": headline_layer,
        "validation_accuracy": float(
            layer_table.loc[
                layer_table["layer"] == headline_layer, "validation_accuracy"
            ].iloc[0]
        ),
        "validation_log_loss": float(
            layer_table.loc[
                layer_table["layer"] == headline_layer, "validation_log_loss"
            ].iloc[0]
        ),
        "overall_test_probe_accuracy": overall_probe_accuracy,
        "overall_test_probe_log_loss": float(
            log_loss(headline_frame["label"], overall_probe_probability)
        ),
        "overall_test_model_accuracy": overall_model_accuracy,
        "probe_accuracy_by_context": context_summary.set_index("ctx_size")[
            "probe_accuracy"
        ].to_dict(),
        "headline_auroc_by_context": headline_auroc.set_index("ctx_size")[
            "auroc"
        ].to_dict(),
        "model_accuracy_by_context": context_summary.set_index("ctx_size")[
            "model_accuracy"
        ].to_dict(),
        "model_wrong_probe_correct": {
            "overall_count": len(wrong_model_right_probe),
            "overall_fraction": float(
                len(wrong_model_right_probe) / len(headline_frame)
            ),
            "long_context_count": len(long_interesting),
            "long_context_fraction": float(len(long_interesting) / len(long_context)),
        },
    }
)